In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load Image 5 (Cameraman with directional motion blur)
image_path = r"C:\Users\Dev Radia\Pictures\Screenshots\image5.png"
raw_image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

if raw_image is None:
    print("Error: Image not found. Check the path!")
else:
    # Ensure proper conversion if image contains an alpha/transparency channel
    if len(raw_image.shape) == 3 and raw_image.shape[2] == 4:
        image = cv2.cvtColor(raw_image, cv2.COLOR_BGRA2GRAY)
    elif len(raw_image.shape) == 3:
        image = cv2.cvtColor(raw_image, cv2.COLOR_BGR2GRAY)
    else:
        image = raw_image

    # Normalize dynamic range
    norm_image = cv2.normalize(image, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)

    # 1. Gentle Smoothing: Low-diameter bilateral filter to stabilize background noise
    # without further smearing motion-degraded edges
    smoothed = cv2.bilateralFilter(norm_image, d=5, sigmaColor=50, sigmaSpace=50)

    # 2. High-Boost / Spatial Sharpening: Second-derivative Laplacian kernel
    # to counter motion smearing and accentuate vertical and diagonal boundaries
    sharpening_kernel = np.array([
        [-1, -1, -1],
        [-1,  9, -1],
        [-1, -1, -1]
    ], dtype=np.float32)
    sharpened_laplacian = cv2.filter2D(smoothed, -1, sharpening_kernel)

    # 3. Unsharp Masking for enhanced gradient recovery
    gaussian_blur = cv2.GaussianBlur(smoothed, (5, 5), 1.5)
    unsharp_mask = cv2.addWeighted(smoothed, 1.8, gaussian_blur, -0.8, 0)

    # Display results
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(norm_image, cmap="gray")
    plt.title("Original (Motion Blurred)")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(sharpened_laplacian, cmap="gray")
    plt.title("High-Boost Sharpened (Laplacian)")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(unsharp_mask, cmap="gray")
    plt.title("Enhanced Unsharp Masking")
    plt.axis("off")

    plt.tight_layout()
    plt.show()